# Post-training checks on the 0.915 model (no training): C = Precise BN, D = learned TTA weights

Runtime > Change runtime type > GPU (L4 is fine), then Runtime > Run all. A few minutes.

- **C. Precise BN** (Wu & Johnson 2021; the BN step of SWA, Izmailov et al. 2018): recompute BatchNorm statistics of the same weights from one pass over the 45k **training** images (clean, or crop+flip). No weights change. Allowed: training images only.
- **D. Learned TTA weights** (Shanmugam et al., ICCV 2021): one learned weight per view (image, mirror, 1-px shifts), fitted on val and judged by 2-fold cross-fitting on val.
- Decisions are made on **val** only (a change must win by > 0.2 pt, 5 of 2,500 images); the internal test is printed as the honest check. A new `submission.csv` is written only if something wins.
- Nothing here uses the hidden test images except to predict them at the very end.
- The first table's baseline row must read val 0.9852 / internal test 0.9812 (the 0.915 model).


## 1. Paths, Drive, device

In [1]:
# ================= Post-training checks on the 0.915 model (no training) =================
import os, re, glob, time, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- paths: set RUN_NAME if you know it; otherwise the run whose submission has internal test 0.9812 is found automatically
RUN_NAME = None                       # e.g. "kd_sewrn_only_student_..." (the 0.915 run)
FIND_TEST_TAG = "test0.9812"          # used only when RUN_NAME is None
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive; drive.mount("/content/drive")
    ROOT, WORK_DIR = "/content/drive/MyDrive/cifar10_resnet_challenge", "/content/work"
except ImportError:
    IN_COLAB = False
    ROOT = os.environ.get("TTA_ROOT", "/scratch/ds8223/cifar10_resnet_challenge"); WORK_DIR = os.environ.get("TTA_WORK", "/scratch/ds8223/kd_work")
CACHE_DIR, SPLIT_DIR = os.path.join(ROOT, "data", "cache"), os.path.join(ROOT, "data", "splits")

if RUN_NAME is None:
    hits = sorted({os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(ROOT, "runs", "*", f"submission_*{FIND_TEST_TAG}*.csv"))})
    print("runs with a submission tagged", FIND_TEST_TAG, ":", hits)
    assert len(hits) == 1, "found 0 or several runs: set RUN_NAME by hand (see the list above)"
    RUN_NAME = hits[0]
RUN_DIR = os.path.join(ROOT, "runs", RUN_NAME)
BEST_CKPT = os.path.join(RUN_DIR, "best.pt")
print("run:", RUN_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow: use a GPU)")
AMP_DTYPE = torch.bfloat16 if device.type == "cpu" or torch.cuda.is_bf16_supported() else None

Mounted at /content/drive
runs with a submission tagged test0.9812 : ['kd_sewrn_only_student_resnext29_slasd_400ep_v1']
run: kd_sewrn_only_student_resnext29_slasd_400ep_v1
device: NVIDIA A100-SXM4-40GB


## 2. Model code (same classes as the training notebooks)

In [2]:
# ---- Models (same code as the training notebooks): ResNet, ResNeXt-29, Shake-Shake, chosen with CFG["ARCH"] ----
class BasicBlock(nn.Module):
    """Post-activation basic block (our original ResNet)."""
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class CifarResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256), blocks=(3, 3, 3), num_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, cin = [], widths[0]
        for i, (w, nb) in enumerate(zip(widths, blocks)):
            for j in range(nb):
                layers.append(BasicBlock(cin, w, 2 if (j == 0 and i > 0) else 1)); cin = w
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn2.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class ResNeXtBlock(nn.Module):
    """ResNeXt bottleneck: 1x1 conv -> grouped 3x3 conv (cardinality groups) -> 1x1 conv, plus skip (Xie et al. 2017)."""
    def __init__(self, cin, inner, cout, stride, cardinality):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, inner, 1, bias=False); self.bn1 = nn.BatchNorm2d(inner)
        self.conv2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm2d(inner)
        self.conv3 = nn.Conv2d(inner, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class ResNeXt29(nn.Module):
    """CIFAR ResNeXt-29: 3x3 stem (64) + 3 stages x 3 blocks; output widths 256/512/1024;
    inner width = cardinality * bottleneck_width, doubled at each stage. (2, 64) is the standard 2x64d."""
    def __init__(self, cardinality=2, bottleneck_width=64, blocks=(3, 3, 3), out_widths=(256, 512, 1024),
                 num_classes=10, dropout=0.0, stem=64, strides=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem, 3, 1, 1, bias=False), nn.BatchNorm2d(stem), nn.ReLU(inplace=True))
        layers, cin = [], stem
        for i, (cout, nb) in enumerate(zip(out_widths, blocks)):
            inner = cardinality * bottleneck_width * (2 ** i)
            for j in range(nb):
                layers.append(ResNeXtBlock(cin, inner, cout, strides[i] if j == 0 else 1, cardinality)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn3.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class SE(nn.Module):
    """Squeeze-excitation (Hu et al. 2018): channel weights from global average pooling. Teachers only."""
    def __init__(self, c, r=16):
        super().__init__()
        self.fc1, self.fc2 = nn.Conv2d(c, c // r, 1), nn.Conv2d(c // r, c, 1)
    def forward(self, x):
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(F.adaptive_avg_pool2d(x, 1)))))

class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop, se_r=0):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
        if se_r: self.se = SE(cout, se_r)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        o = self.conv2(self.dropout(F.relu(self.bn2(self.conv1(o)))))
        if hasattr(self, "se"): o = self.se(o)
        return o + sc

class WRN(nn.Module):
    """WideResNet for CIFAR (Zagoruyko & Komodakis 2016). k = widen factor (WRN-28-10: 36.5M params) or explicit stage widths."""
    def __init__(self, depth=28, k=10, dropout=0.0, num_classes=10, se_r=0):
        super().__init__()
        n = (depth - 4) // 6
        w = [16] + (list(k) if isinstance(k, (tuple, list)) else [16 * k, 32 * k, 64 * k])   # k: widen factor, or the 3 stage widths
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, dropout, se_r)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(0.0); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))

def _build_base(c):
    """Build any of our architectures from a config dict (also rebuilds teachers from their saved cfg)."""
    arch = c.get("ARCH", "resnet")
    if arch == "resnet":
        return CifarResNet(c["WIDTHS"], c["BLOCKS"], dropout=c.get("DROPOUT", 0.0))
    if arch == "resnext":
        return ResNeXt29(c.get("CARDINALITY", 2), c.get("BOTTLENECK_WIDTH", 64), c.get("BLOCKS", (3, 3, 3)),
                         c.get("WIDTHS", (256, 512, 1024)), dropout=c.get("DROPOUT", 0.0),
                         strides=tuple(c.get("STRIDES", (1, 2, 2))))   # (1, 1, 2) = higher-resolution student, same params
    if arch in ("wrn", "sewrn"):     # sewrn = WRN with SE blocks: a training-only teacher
        return WRN(c["DEPTH"], c["WIDEN"], c.get("DROPOUT", 0.0), se_r=c.get("SE_REDUCTION", 16) if arch == "sewrn" else 0)
    raise ValueError(f"unknown ARCH {arch}")

def build_model(c):
    """Base network; with SLA on, adds the joint (class x rotation) head used only during training."""
    net = _build_base(c)
    if c.get("SLA", False):
        net.fc_joint = nn.Linear(net.fc.in_features, 10 * c.get("SLA_ROTATIONS", 4))
    return net

def forward_both(net, x):
    """Returns (10-class self-distillation head logits, joint class x rotation head logits)."""
    f = net.drop(net.features(x))
    return net.fc(f), net.fc_joint(f)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Load the 0.915 best.pt, data and the same split

In [3]:
# ---- checkpoint, data, split (same split file as every run)
best = torch.load(BEST_CKPT, map_location=device, weights_only=False)
CFG = best["cfg"]
X_full = np.load(os.path.join(CACHE_DIR, "train_x.npy")); y_full = np.load(os.path.join(CACHE_DIR, "train_y.npy"))
X_test = np.load(os.path.join(CACHE_DIR, "test_x.npy")); test_ids = np.load(os.path.join(CACHE_DIR, "test_ids.npy"))
SAMPLE_SUB = pd.read_csv(os.path.join(CACHE_DIR, "sample_submission.csv"))
sp = np.load(os.path.join(SPLIT_DIR, f"split_seed{CFG['SPLIT_SEED']}_val{CFG['VAL_FRAC']}_test{CFG['TEST_FRAC']}_n{len(y_full)}.npz"))
tr_idx, va_idx, te_idx = sp["train"], sp["val"], sp["test"]
def to_dev(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(device)
Xva, yva = to_dev(X_full[va_idx]), torch.from_numpy(y_full[va_idx]).to(device)
Xte_int, yte_int = to_dev(X_full[te_idx]), torch.from_numpy(y_full[te_idx]).to(device)
Xkaggle = to_dev(X_test)
Xtr = to_dev(X_full[tr_idx])                           # 45k TRAINING images: used only to recompute BN statistics
MEAN, STD = best["norm"]["mean"].to(device), best["norm"]["std"].to(device)   # exact normalisation of that run
def normalize(x_u8): return ((x_u8.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def _ag_logits(m, xb):
    """SLA+AG: average the joint-head logit of (class, k) on the image rotated by k*90 degrees."""
    agg = 0
    for k in range(CFG.get("SLA_ROTATIONS", 4)):
        _, j = forward_both(m, torch.rot90(xb, k, dims=(2, 3)))
        agg = agg + j.float().view(len(xb), 10, -1)[:, :, k]
    return agg / CFG.get("SLA_ROTATIONS", 4)

net = build_model(CFG).to(device).to(memory_format=torch.channels_last)
net.load_state_dict(best["model"]); net.eval()
has_joint = getattr(net, "fc_joint", None) is not None
print(f"params {count_params(net):,} | saved val {best['val_acc']:.4f} | SLA head: {has_joint} | val {len(Xva)} | internal test {len(Xte_int)} | kaggle {len(Xkaggle)} | train {len(Xtr)}")

params 4,998,738 | saved val 0.9856 | SLA head: True | val 2500 | internal test 2500 | kaggle 2000 | train 45000


## 4. Prediction helper (flip TTA as in the 0.915 submission)

In [4]:
# ---- shared: prediction with flip TTA exactly as the 0.915 submission (sum of softmax over image + mirror) ----
import copy
@torch.no_grad()
def predict(m, X, mode="sd", bs=500):
    m.eval(); out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs]); p = 0
        for v in (xb, xb.flip(3)):
            with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                if mode == "sd":   q = m(v).float().softmax(1)
                elif mode == "ag": q = _ag_logits(m, v).float().softmax(1)
                else:              q = 0.5 * (m(v).float().softmax(1) + _ag_logits(m, v).float().softmax(1))
            p = p + q
        out.append(p)
    return torch.cat(out)
def acc(p, y): return (p.argmax(1) == y).float().mean().item()
modes = ("sd", "ag", "sdag") if has_joint else ("sd",)

## 5. C: Precise BN on training images

In [5]:
# ---- C. Precise BN (Wu & Johnson 2021; also the BN step of SWA, Izmailov et al. 2018) ----
# The saved weights are EMA weights; their BN running statistics were averaged along with them, and during training BN saw
# batches of 4 rotations + CutMix + TrivialAugment. Here BN statistics are recomputed for the SAME weights from one pass over the
# 45k TRAINING images (never val, internal test or the hidden test), upright, as an exact average over all images.
def crop_flip(x):
    n, ar = len(x), torch.arange(32, device=x.device)
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    dy, dx = torch.randint(0, 9, (n,), device=x.device), torch.randint(0, 9, (n,), device=x.device)
    x = xp[torch.arange(n, device=x.device)[:, None, None, None], torch.arange(3, device=x.device)[None, :, None, None],
           (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    flip = torch.rand(n, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def precise_bn(model, X, augment, bs=256, seed=0):
    m = copy.deepcopy(model)
    bns = [b for b in m.modules() if isinstance(b, nn.modules.batchnorm._BatchNorm)]
    for b in bns: b.reset_running_stats(); b.momentum = None          # momentum=None -> exact cumulative average
    torch.manual_seed(seed); perm = torch.randperm(len(X), device=X.device)
    m.train()
    for i in range(0, len(X), bs):
        xb = normalize(X[perm[i:i + bs]])
        if augment: xb = crop_flip(xb)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            m(xb)
    m.eval(); return m

t0 = time.time()
MODELS = {"original (0.915 submission)": net,
          "precise BN, clean train images": precise_bn(net, Xtr, augment=False),
          "precise BN, crop+flip train images": precise_bn(net, Xtr, augment=True)}
rows = []
for name, m in MODELS.items():
    for md in modes:
        rows.append({"model": name, "mode": md, "val": acc(predict(m, Xva, md), yva), "test": acc(predict(m, Xte_int, md), yte_int)})
resC = pd.DataFrame(rows)
print(f"done in {time.time() - t0:.0f}s\n")
print(resC.assign(val=resC.val.round(4), test=resC.test.round(4)).to_string(index=False))

# baseline = how the 0.915 submission was made (sd unless ag/sdag won VAL by > 0.2 pt); a change must beat it on VAL by > 0.2 pt
b = resC[resC.model == "original (0.915 submission)"].set_index("mode")
alt = max([m for m in modes if m != "sd"], key=lambda m: b.loc[m, "val"], default="sd")
base_mode = alt if alt != "sd" and b.loc[alt, "val"] > b.loc["sd", "val"] + 0.002 else "sd"
baseC = resC[(resC.model == "original (0.915 submission)") & (resC["mode"] == base_mode)].iloc[0]
candC = resC[resC.model != "original (0.915 submission)"].sort_values("val", ascending=False).iloc[0]
chosenC = candC if candC.val > baseC.val + 0.002 else baseC
print(f"\nBASELINE: {baseC.model}, '{baseC['mode']}': val {baseC.val:.4f} | internal test {baseC.test:.4f}   (expected 0.9852 / 0.9812)")
print(f"BEST PRECISE BN ON VAL: {candC.model}, '{candC['mode']}': val {candC.val:.4f} | internal test {candC.test:.4f} "
      f"-> internal test change {100 * (candC.test - baseC.test):+.2f} pt ({int(round((candC.test - baseC.test) * 2500)):+d} images)")
print("C DECISION:", f"use '{chosenC.model}'" if chosenC.model != baseC.model else "keep the original BN statistics (no VAL gain > 0.2 pt)")
M_C, MODE_C = MODELS[chosenC.model], chosenC["mode"]
if chosenC.model != baseC.model:
    torch.save({"model": M_C.state_dict(), "cfg": CFG, "norm": best["norm"], "note": f"0.915 weights, {chosenC.model}"},
               os.path.join(RUN_DIR, "best_preciseBN.pt"))
    print("saved", os.path.join(RUN_DIR, "best_preciseBN.pt"))

done in 29s

                             model mode    val   test
       original (0.915 submission)   sd 0.9852 0.9812
       original (0.915 submission)   ag 0.9828 0.9808
       original (0.915 submission) sdag 0.9836 0.9808
    precise BN, clean train images   sd 0.9676 0.9660
    precise BN, clean train images   ag 0.9708 0.9684
    precise BN, clean train images sdag 0.9760 0.9720
precise BN, crop+flip train images   sd 0.9692 0.9660
precise BN, crop+flip train images   ag 0.9752 0.9732
precise BN, crop+flip train images sdag 0.9768 0.9776

BASELINE: original (0.915 submission), 'sd': val 0.9852 | internal test 0.9812   (expected 0.9852 / 0.9812)
BEST PRECISE BN ON VAL: precise BN, crop+flip train images, 'sdag': val 0.9768 | internal test 0.9776 -> internal test change -0.36 pt (-9 images)
C DECISION: keep the original BN statistics (no VAL gain > 0.2 pt)


## 6. D: learned TTA weights (cross-fitted on val)

In [6]:
# ---- D. Learned TTA weights (Shanmugam et al., ICCV 2021, "AugTTA": one non-negative weight per view) ----
# Views: image, mirror, 1-px shifts down-right / up-left with mirrors (6). Weights are learned on VAL; to judge them honestly
# without touching the internal test, they are 2-fold CROSS-FITTED on val (learn on one half, score the other half).
def shift(x, dy, dx, p=2):
    if dy == 0 and dx == 0: return x
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    return xp[:, :, p - dy:p - dy + 32, p - dx:p - dx + 32].contiguous(memory_format=torch.channels_last)
VIEWS = [(0, 0, False), (0, 0, True), (1, 1, False), (1, 1, True), (-1, -1, False), (-1, -1, True)]
@torch.no_grad()
def view_logits(m, X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs]); vs = []
        for dy, dx, fl in VIEWS:
            v = shift(xb.flip(3) if fl else xb, dy, dx)
            with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                vs.append(m(v).float())
        out.append(torch.stack(vs, 1))                     # (batch, 6 views, 10)
    return torch.cat(out)
def fit_weights(L, y, steps=300):
    a = torch.zeros(L.shape[1], device=L.device, requires_grad=True)
    opt = torch.optim.Adam([a], lr=0.05)
    for _ in range(steps):
        w = a.softmax(0); loss = F.cross_entropy((L * w[None, :, None]).sum(1), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return a.detach().softmax(0)
def tta_pred(L, w): return (L * w[None, :, None]).sum(1)

t0 = time.time()
Lva, Lte, Lkg = view_logits(M_C, Xva), view_logits(M_C, Xte_int), view_logits(M_C, Xkaggle)
g = torch.Generator().manual_seed(0); idx = torch.randperm(len(yva), generator=g).to(device); h = len(idx) // 2
folds = [(idx[:h], idx[h:]), (idx[h:], idx[:h])]
correct = 0
for fit_i, score_i in folds:
    w = fit_weights(Lva[fit_i], yva[fit_i])
    correct += (tta_pred(Lva[score_i], w).argmax(1) == yva[score_i]).sum().item()
cv_acc = correct / len(yva)
w_all = fit_weights(Lva, yva)
d_test = (tta_pred(Lte, w_all).argmax(1) == yte_int).float().mean().item()
ref_val = acc(predict(M_C, Xva, MODE_C), yva); ref_test = acc(predict(M_C, Xte_int, MODE_C), yte_int)
print(f"done in {time.time() - t0:.0f}s")
print("learned view weights (image, mirror, +1px, +1px mirror, -1px, -1px mirror):", [round(v, 3) for v in w_all.tolist()])
print(f"current setting ('{MODE_C}', flip): val {ref_val:.4f} | internal test {ref_test:.4f}")
print(f"learned TTA: CROSS-FITTED val {cv_acc:.4f} | internal test {d_test:.4f} -> internal test change {100 * (d_test - ref_test):+.2f} pt")
use_D = cv_acc > ref_val + 0.002
print("D DECISION:", "use learned TTA weights" if use_D else "keep the current TTA (no cross-fitted VAL gain > 0.2 pt)")

done in 11s
learned view weights (image, mirror, +1px, +1px mirror, -1px, -1px mirror): [0.12, 0.267, 0.275, 0.004, 0.056, 0.277]
current setting ('sd', flip): val 0.9852 | internal test 0.9812
learned TTA: CROSS-FITTED val 0.9848 | internal test 0.9824 -> internal test change +0.12 pt
D DECISION: keep the current TTA (no cross-fitted VAL gain > 0.2 pt)


## 7. Submission only if C or D won on val

In [7]:
# ---- final: write a submission only if C or D changed something; otherwise keep the 0.915 submission ----
base_pred = predict(net, Xkaggle, base_mode).argmax(1)
if use_D:                 final, desc = tta_pred(Lkg, w_all).argmax(1), f"learned TTA on {chosenC.model}"
elif chosenC.model != baseC.model: final, desc = predict(M_C, Xkaggle, MODE_C).argmax(1), f"{chosenC.model}, mode {MODE_C}"
else:                     final, desc = None, None
if final is None:
    print("Nothing beat the 0.915 setup on VAL -> keep the 0.915 submission. No new file written.")
else:
    kp = final.cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = re.sub(r"[^a-z0-9]+", "_", desc.lower()).strip("_")[:60]
    drive_sub = os.path.join(RUN_DIR, f"submission_posthoc_{tag}.csv"); sub.to_csv(drive_sub, index=False)
    os.makedirs(os.path.join(WORK_DIR, "posthoc"), exist_ok=True)
    local_sub = os.path.join(WORK_DIR, "posthoc", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"NEW SUBMISSION ({desc}) saved: {drive_sub}")
    print(f"Kaggle predictions that differ from the 0.915 submission: {(final != base_pred).sum().item()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)

Nothing beat the 0.915 setup on VAL -> keep the 0.915 submission. No new file written.
